# Train And MFR Combined Notebook

This notebook combines training, adapter download/reuse, MFR strategy sweeps, and final Codabench ZIP creation.


## 1. Setup Folder And Runtime Aliases


In [ ]:
from pathlib import Path
import os
import shutil

def looks_like_submission_root(path):
    return (
        (path / "01. train_lora.py").exists()
        and (path / "01. predict_lora.py").exists()
        and (path / "01. utils.py").exists()
        and (path / "01. requirements.txt").exists()
    )

def find_submission_root():
    starts = []
    configured = os.environ.get("MULTILEXNORM_SUBMISSION_ROOT")
    if configured:
        starts.append(Path(configured).expanduser())
    starts.append(Path.cwd())

    checked = set()
    for start in starts:
        for base in [start, *start.parents]:
            for candidate in [base, base / "MultiLexNorm_Submission"]:
                key = str(candidate)
                if key in checked:
                    continue
                checked.add(key)
                if looks_like_submission_root(candidate):
                    return candidate.resolve()

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Colab Drive mount skipped:", repr(exc))

    for drive_root in [Path("drive") / "MyDrive", Path("../drive") / "MyDrive"]:
        if drive_root.exists():
            candidate = drive_root / "MultiLexNorm_Submission"
            if looks_like_submission_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the submission folder. Run this notebook from inside "
        "MultiLexNorm_Submission, from its parent folder, or set "
        "MULTILEXNORM_SUBMISSION_ROOT to the folder path."
    )

SUBMISSION_ROOT = find_submission_root()

RUNTIME_DIR = SUBMISSION_ROOT / "_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

ALIASES = {
    "01. train_lora.py": "train_lora.py",
    "01. predict_lora.py": "predict_lora.py",
    "01. utils.py": "utils.py",
    "01. requirements.txt": "requirements.txt",
}
for source_name, target_name in ALIASES.items():
    source_path = SUBMISSION_ROOT / source_name
    target_path = RUNTIME_DIR / target_name
    if not source_path.exists():
        raise FileNotFoundError(source_path)
    if not target_path.exists() or source_path.read_bytes() != target_path.read_bytes():
        shutil.copy2(source_path, target_path)

os.chdir(RUNTIME_DIR)
print("Submission root:", SUBMISSION_ROOT)
print("Runtime directory:", RUNTIME_DIR)
print("Working directory:", Path.cwd())


## 2. Install Requirements


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print("Installed requirements from", Path("requirements.txt").resolve())


## 3. Patch Runtime Scripts For Gemma 4


In [ ]:
from pathlib import Path


def replace_once(text, old, new, label):
    if old not in text:
        print(f"Patch target not found or already changed: {label}")
        return text
    return text.replace(old, new, 1)


def patch_train_lora_for_gemma4(path=Path("train_lora.py")):
    text = path.read_text(encoding="utf-8")
    if "load_tokenizer_for_training" in text and "AutoModelForImageTextToText" in text:
        print("train_lora.py already has explicit Gemma 4 support")
        return

    text = replace_once(
        text,
        """def resolve_model_family(args: argparse.Namespace, model: Any) -> str:\n""",
        """def is_gemma4_model_id(model_id: str) -> bool:\n    normalized = model_id.lower().replace("_", "-")\n    return "gemma-4" in normalized or "gemma4" in normalized\n\n\ndef wants_gemma4_loader(args: argparse.Namespace) -> bool:\n    if args.model_family == "gemma4":\n        return True\n    if args.model_family == "generic":\n        return False\n    return is_gemma4_model_id(args.model_id)\n\n\ndef resolve_model_family(args: argparse.Namespace, model: Any) -> str:\n""",
        "train_lora helper insertion",
    )
    text = replace_once(
        text,
        """    return [item.strip() for item in target_spec.split(",") if item.strip()]\n\n\ndef parse_json_list(text: str) -> list[Any] | None:\n""",
        """    return [item.strip() for item in target_spec.split(",") if item.strip()]\n\n\ndef load_tokenizer_for_training(args: argparse.Namespace) -> Any:\n    from transformers import AutoProcessor, AutoTokenizer\n\n    if wants_gemma4_loader(args):\n        processor = AutoProcessor.from_pretrained(args.model_id, token=args.hf_token)\n        tokenizer = getattr(processor, "tokenizer", processor)\n        if getattr(tokenizer, "chat_template", None) is None and getattr(processor, "chat_template", None):\n            tokenizer.chat_template = processor.chat_template\n        print("Tokenizer loader: AutoProcessor.tokenizer for Gemma 4 text-only MultiLexNorm training")\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_id, token=args.hf_token)\n        print("Tokenizer loader: AutoTokenizer")\n\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    return tokenizer\n\n\ndef load_base_model_for_training(args: argparse.Namespace, compute_dtype: Any, quantization_config: Any) -> Any:\n    from transformers import AutoModelForCausalLM, AutoModelForImageTextToText\n\n    loader_cls = AutoModelForImageTextToText if wants_gemma4_loader(args) else AutoModelForCausalLM\n    print("Model loader:", loader_cls.__name__)\n    return loader_cls.from_pretrained(\n        args.model_id,\n        device_map="auto",\n        dtype=compute_dtype,\n        quantization_config=quantization_config,\n        token=args.hf_token,\n    )\n\n\ndef parse_json_list(text: str) -> list[Any] | None:\n""",
        "train_lora loader insertion",
    )
    text = replace_once(
        text,
        """    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer\n""",
        """    from transformers import BitsAndBytesConfig, Trainer\n""",
        "train_lora import replacement",
    )
    text = replace_once(
        text,
        """    tokenizer = AutoTokenizer.from_pretrained(args.model_id, token=args.hf_token)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n""",
        """    tokenizer = load_tokenizer_for_training(args)\n""",
        "train_lora tokenizer loader replacement",
    )
    text = replace_once(
        text,
        """    model = AutoModelForCausalLM.from_pretrained(\n        args.model_id,\n        device_map="auto",\n        dtype=compute_dtype,\n        quantization_config=quantization_config,\n        token=args.hf_token,\n    )\n""",
        """    model = load_base_model_for_training(args, compute_dtype, quantization_config)\n""",
        "train_lora model loader replacement",
    )
    path.write_text(text, encoding="utf-8")
    print("Patched train_lora.py for explicit Gemma 4 support")


def patch_predict_lora_for_gemma4(path=Path("predict_lora.py")):
    text = path.read_text(encoding="utf-8")
    if "AutoModelForImageTextToText" in text and "is_gemma4_model_id" in text:
        print("predict_lora.py already has explicit Gemma 4 support")
        return

    text = replace_once(
        text,
        """def load_model_and_tokenizer(args: argparse.Namespace, model_id: str) -> tuple[Any, Any]:\n""",
        """def is_gemma4_model_id(model_id: str) -> bool:\n    normalized = model_id.lower().replace("_", "-")\n    return "gemma-4" in normalized or "gemma4" in normalized\n\n\ndef load_model_and_tokenizer(args: argparse.Namespace, model_id: str) -> tuple[Any, Any]:\n""",
        "predict_lora helper insertion",
    )
    text = replace_once(
        text,
        """    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n    tokenizer = AutoTokenizer.from_pretrained(model_id, token=args.hf_token)\n""",
        """    from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor, AutoTokenizer, BitsAndBytesConfig\n\n    if is_gemma4_model_id(model_id):\n        processor = AutoProcessor.from_pretrained(model_id, token=args.hf_token)\n        tokenizer = getattr(processor, "tokenizer", processor)\n        if getattr(tokenizer, "chat_template", None) is None and getattr(processor, "chat_template", None):\n            tokenizer.chat_template = processor.chat_template\n        model_loader = AutoModelForImageTextToText\n        print("Model loader: AutoModelForImageTextToText for Gemma 4 text-only inference")\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(model_id, token=args.hf_token)\n        model_loader = AutoModelForCausalLM\n        print("Model loader: AutoModelForCausalLM")\n""",
        "predict_lora loader replacement",
    )
    text = replace_once(
        text,
        """    base_model = AutoModelForCausalLM.from_pretrained(\n""",
        """    base_model = model_loader.from_pretrained(\n""",
        "predict_lora model class replacement",
    )
    path.write_text(text, encoding="utf-8")
    print("Patched predict_lora.py for explicit Gemma 4 support")


patch_train_lora_for_gemma4()
patch_predict_lora_for_gemma4()


## 4. Hugging Face Login


In [ ]:
import getpass
import os
from huggingface_hub import login, whoami

HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face token: ")

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

try:
    print(whoami())
except Exception as exc:
    print("HF login check failed:", repr(exc))


## 5. Set Adapter Source

Set the Hugging Face adapter repo and choose whether to train a new adapter.


In [ ]:
# Hugging Face repo containing adapter_config.json and adapter_model.safetensors.
HF_ADAPTER_REPO_ID = "qwfjop/multilexnorm-gemma4-lora"

# Leave blank if the adapter files are at the repo root.
HF_ADAPTER_SUBFOLDER = ""

# False: download and use the uploaded adapter. True: train a new adapter.
TRAIN_NEW_ADAPTER = False


## 6. Adapter Readiness Check And Run Configuration


In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import shutil

RUN_ID = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%y%m%d_%H%M")
EXPERIMENT_ID = RUN_ID

DATASET_PATH = SUBMISSION_ROOT / "data" / "multilexnorm2026-dev-pub"
DATASET_ID = "weerayut/multilexnorm2026-dev-pub"

GEMMA4_MODEL_KEY = "e4b"
GEMMA4_MODELS = {
    "e4b": {
        "model_id": "google/gemma-4-E4B-it",
        "max_length": 768,
        "epochs": 2,
        "batch_size": 8,
        "grad_accum_steps": 2,
        "eval_batch_size": 16,
        "learning_rate": 5e-5,
        "lora_r": 32,
        "lora_alpha": 64,
        "save_steps": 1000,
        "eval_steps": 500,
    },
    "26b_a4b": {
        "model_id": "google/gemma-4-26B-A4B-it",
        "max_length": 512,
        "epochs": 1,
        "batch_size": 2,
        "grad_accum_steps": 12,
        "eval_batch_size": 4,
        "learning_rate": 3e-5,
        "lora_r": 8,
        "lora_alpha": 16,
        "save_steps": 1000,
        "eval_steps": 500,
    },
}
CFG = GEMMA4_MODELS[GEMMA4_MODEL_KEY]
MODEL_ID = CFG["model_id"]

ADAPTER_DIR = RUNTIME_DIR / "adapter"
ADAPTER_DOWNLOAD_DIR = RUNTIME_DIR / "adapter_download"
TRAINED_ADAPTER_ROOT = SUBMISSION_ROOT / "adapter_runs"
BASELINE_RUN_OUTPUT_ROOT = SUBMISSION_ROOT / "outputs" / "combined_cache" / RUN_ID
OUTPUT_ROOT = SUBMISSION_ROOT / "outputs" / "train_and_mfr" / RUN_ID
VALIDATION_OUTPUT_ROOT = OUTPUT_ROOT / "validation_predictions"
TEST_OUTPUT_ROOT = OUTPUT_ROOT / "test_predictions"
for path in [ADAPTER_DIR, ADAPTER_DOWNLOAD_DIR, TRAINED_ADAPTER_ROOT, BASELINE_RUN_OUTPUT_ROOT, OUTPUT_ROOT, VALIDATION_OUTPUT_ROOT, TEST_OUTPUT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

HF_ADAPTER_REPO_ID = (globals().get("HF_ADAPTER_REPO_ID", "") or os.environ.get("HF_ADAPTER_REPO_ID", "")).strip()
HF_ADAPTER_SUBFOLDER = (globals().get("HF_ADAPTER_SUBFOLDER", "") or os.environ.get("HF_ADAPTER_SUBFOLDER", "")).strip().strip("/")
TRAIN_NEW_ADAPTER = bool(globals().get("TRAIN_NEW_ADAPTER", False))

HF_ADAPTER_FILES = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
    "train_lora_args.json",
    "README.md",
]

def adapter_readiness(adapter_dir):
    adapter_dir = Path(adapter_dir)
    adapter_model = adapter_dir / "adapter_model.safetensors"
    required = {
        "adapter_config.json": (adapter_dir / "adapter_config.json").exists(),
        "adapter_model.safetensors": adapter_model.exists(),
    }
    optional = {
        "tokenizer": (adapter_dir / "tokenizer.json").exists() or (adapter_dir / "tokenizer_config.json").exists(),
        "chat_template.jinja": (adapter_dir / "chat_template.jinja").exists(),
        "train_lora_args.json": (adapter_dir / "train_lora_args.json").exists(),
    }
    model_size = adapter_model.stat().st_size if adapter_model.exists() else 0
    looks_like_pointer = adapter_model.exists() and model_size < 1_000_000
    ready = all(required.values()) and not looks_like_pointer
    return {
        "ready": ready,
        "required": required,
        "optional": optional,
        "adapter_model_size_mb": model_size / (1024 * 1024),
        "looks_like_pointer": looks_like_pointer,
    }

def download_adapter_from_hf(repo_id, adapter_dir, subfolder=""):
    if not repo_id:
        raise ValueError("Set HF_ADAPTER_REPO_ID or set TRAIN_NEW_ADAPTER=True.")

    from huggingface_hub import snapshot_download

    token = HF_TOKEN if "HF_TOKEN" in globals() and HF_TOKEN else None
    adapter_dir = Path(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)

    patterns = [f"{subfolder}/{name}" if subfolder else name for name in HF_ADAPTER_FILES]
    print("Downloading adapter from Hugging Face Hub:", repo_id)
    if subfolder:
        print("Adapter subfolder:", subfolder)

    snapshot_download(
        repo_id=repo_id,
        local_dir=str(ADAPTER_DOWNLOAD_DIR),
        allow_patterns=patterns,
        token=token,
    )

    source_dir = ADAPTER_DOWNLOAD_DIR / subfolder if subfolder else ADAPTER_DOWNLOAD_DIR
    for name in HF_ADAPTER_FILES:
        source_path = source_dir / name
        if source_path.exists():
            shutil.copy2(source_path, adapter_dir / name)

    status = adapter_readiness(adapter_dir)
    if not status["ready"]:
        raise FileNotFoundError(
            "Adapter download did not produce required files. Expected "
            "adapter_config.json and adapter_model.safetensors."
        )
    return status

if TRAIN_NEW_ADAPTER:
    ADAPTER_STATUS = {"ready": False, "required": {}, "optional": {}, "adapter_model_size_mb": 0.0, "looks_like_pointer": False}
    VALIDATION_ADAPTER_DIR = TRAINED_ADAPTER_ROOT / f"{GEMMA4_MODEL_KEY}_train_only_{RUN_ID}"
    FINAL_ADAPTER_DIR = TRAINED_ADAPTER_ROOT / f"{GEMMA4_MODEL_KEY}_trainval_{RUN_ID}"
    RUN_TRAIN_VALIDATION_ADAPTER = True
    RUN_TRAIN_FINAL_ADAPTER = True
else:
    ADAPTER_STATUS = adapter_readiness(ADAPTER_DIR)
    if not ADAPTER_STATUS["ready"]:
        ADAPTER_STATUS = download_adapter_from_hf(HF_ADAPTER_REPO_ID, ADAPTER_DIR, HF_ADAPTER_SUBFOLDER)
    VALIDATION_ADAPTER_DIR = ADAPTER_DIR
    FINAL_ADAPTER_DIR = ADAPTER_DIR
    RUN_TRAIN_VALIDATION_ADAPTER = False
    RUN_TRAIN_FINAL_ADAPTER = False

RUN_VALIDATION_PREDICTIONS = True
RUN_TEST_SUBMISSION = True

VALIDATION_STRATEGIES = ["model", "mfr-known", "mfr-confidence", "lang-best"]
MFR_CONFIDENCE_THRESHOLD = 1.00
FALLBACK = "mfr"
FORCE_RETRAIN = False
FORCE_REPREDICT = False
FAIL_FAST = True

def dataset_arg():
    return str(DATASET_PATH if DATASET_PATH.exists() else DATASET_ID)

print("RUN_ID:", RUN_ID)
print("MODEL_ID:", MODEL_ID)
print("Dataset:", dataset_arg())
print("HF adapter repo:", HF_ADAPTER_REPO_ID or "not set")
print("Adapter directory:", ADAPTER_DIR)
print("Adapter status:", ADAPTER_STATUS)
print("Train new adapter:", TRAIN_NEW_ADAPTER)
print("Validation adapter:", VALIDATION_ADAPTER_DIR)
print("Final/test adapter:", FINAL_ADAPTER_DIR)
print("Output root:", OUTPUT_ROOT)


## 7. Helpers


In [ ]:
import itertools
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from datasets import concatenate_datasets
from train_lora import load_multilexnorm
from predict_lora import make_mfr_counts, mfr_token_confidence, mfr_token_prediction
from utils import counting, evaluate, mfr

def run_cmd(cmd):
    cmd = [str(item) for item in cmd]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

def load_records(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def save_records(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(records, ensure_ascii=False), encoding='utf-8')
    return path

def score_records(records, info=False):
    return evaluate(
        raw=[row['raw'] for row in records],
        gold=[row['norm'] for row in records],
        pred=[row['pred'] for row in records],
        info=info,
    )

def score_row(name, records):
    lai, acc, err = score_records(records, info=False)
    return {'name': name, 'lai': lai, 'accuracy': acc, 'err': err}

def make_zip(predictions_path, zip_path):
    predictions_path = Path(predictions_path)
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(predictions_path, arcname='predictions.json')
    return zip_path

def existing_success(path):
    path = Path(path)
    return (
        (path / "adapter_config.json").exists()
        and (path / "adapter_model.safetensors").exists()
    )


## 8. Train Validation Adapter If Needed


In [ ]:
if RUN_TRAIN_VALIDATION_ADAPTER:
    if existing_success(VALIDATION_ADAPTER_DIR) and not FORCE_RETRAIN:
        print("Validation adapter already exists:", VALIDATION_ADAPTER_DIR)
    else:
        run_cmd([
            sys.executable,
            "train_lora.py",
            "--model-id", MODEL_ID,
            "--model-family", "gemma4",
            "--dataset-id", dataset_arg(),
            "--output-dir", VALIDATION_ADAPTER_DIR,
            "--hf-token", HF_TOKEN,
            "--quantization", "4bit",
            "--max-train-examples", "0",
            "--max-eval-examples", "1000",
            "--max-length", CFG["max_length"],
            "--epochs", CFG["epochs"],
            "--batch-size", CFG["batch_size"],
            "--grad-accum-steps", CFG["grad_accum_steps"],
            "--eval-batch-size", CFG["eval_batch_size"],
            "--learning-rate", CFG["learning_rate"],
            "--lora-r", CFG["lora_r"],
            "--lora-alpha", CFG["lora_alpha"],
            "--lora-dropout", "0.05",
            "--eval-steps", CFG["eval_steps"],
            "--save-steps", CFG["save_steps"],
            "--logging-steps", "20",
            "--dataloader-num-workers", "2",
            "--preview-examples", "5",
        ])
else:
    print("Skipping validation-adapter training because an adapter is already ready:", VALIDATION_ADAPTER_DIR)

print("Validation adapter path:", VALIDATION_ADAPTER_DIR)


## 9. Train Final Adapter If Needed


In [ ]:
if RUN_TRAIN_FINAL_ADAPTER:
    if existing_success(FINAL_ADAPTER_DIR) and not FORCE_RETRAIN:
        print("Final train+validation adapter already exists:", FINAL_ADAPTER_DIR)
    else:
        run_cmd([
            sys.executable,
            "train_lora.py",
            "--model-id", MODEL_ID,
            "--model-family", "gemma4",
            "--dataset-id", dataset_arg(),
            "--output-dir", FINAL_ADAPTER_DIR,
            "--hf-token", HF_TOKEN,
            "--quantization", "4bit",
            "--include-validation-in-train",
            "--max-train-examples", "0",
            "--max-eval-examples", "1000",
            "--max-length", CFG["max_length"],
            "--epochs", CFG["epochs"],
            "--batch-size", CFG["batch_size"],
            "--grad-accum-steps", CFG["grad_accum_steps"],
            "--eval-batch-size", CFG["eval_batch_size"],
            "--learning-rate", CFG["learning_rate"],
            "--lora-r", CFG["lora_r"],
            "--lora-alpha", CFG["lora_alpha"],
            "--lora-dropout", "0.05",
            "--eval-steps", CFG["eval_steps"],
            "--save-steps", CFG["save_steps"],
            "--logging-steps", "20",
            "--dataloader-num-workers", "2",
            "--preview-examples", "0",
        ])
else:
    print("Skipping final-adapter training because an adapter is already ready:", FINAL_ADAPTER_DIR)

print("Final/test adapter path:", FINAL_ADAPTER_DIR)


## 10. Create Or Reuse Validation Model Predictions


In [ ]:
BASELINE_VALIDATION_MODEL_PREDICTIONS = (
    BASELINE_RUN_OUTPUT_ROOT / 'validation_predictions' / f'{GEMMA4_MODEL_KEY}_model_{RUN_ID}' / 'predictions.json'
)
VALIDATION_MODEL_DIR = VALIDATION_OUTPUT_ROOT / 'validation_model_once'
VALIDATION_MODEL_PREDICTIONS = (
    BASELINE_VALIDATION_MODEL_PREDICTIONS
    if BASELINE_VALIDATION_MODEL_PREDICTIONS.exists()
    else VALIDATION_MODEL_DIR / 'predictions.json'
)

if not VALIDATION_MODEL_PREDICTIONS.exists():
    run_cmd([
        sys.executable,
        'predict_lora.py',
        '--adapter-dir', VALIDATION_ADAPTER_DIR,
        '--model-id', MODEL_ID,
        '--dataset-id', dataset_arg(),
        '--split', 'validation',
        '--output-dir', VALIDATION_MODEL_DIR,
        '--hf-token', HF_TOKEN,
        '--quantization', '4bit',
        '--prediction-strategy', 'model',
        '--fallback', 'mfr',
        '--max-new-tokens', '96',
        '--batch-size', '0',
        '--progress-steps', '256',
        '--preview-examples', '5',
    ])
else:
    print('Using cached validation model predictions:', VALIDATION_MODEL_PREDICTIONS)

validation_model_records = load_records(VALIDATION_MODEL_PREDICTIONS)
score_records(validation_model_records, info=True)


## 11. Offline Strategy Functions


In [ ]:
data = load_multilexnorm(dataset_arg())
validation_counts_by_lang = make_mfr_counts(data, 'validation')

def clone_with_pred(records, pred_fn):
    out = []
    for row in records:
        new_row = dict(row)
        new_row['pred'] = pred_fn(row)
        out.append(new_row)
    return out

def pred_raw(row):
    return list(row['raw'])

def pred_mfr(row, counts_by_lang=validation_counts_by_lang):
    return mfr(row['raw'], counts_by_lang.get(row['lang'], {}))

def pred_mfr_known(row, counts_by_lang=validation_counts_by_lang):
    token_counts = counts_by_lang.get(row['lang'], {})
    pred = []
    for raw_token, model_token in zip(row['raw'], row['pred']):
        pred.append(mfr_token_prediction(raw_token, token_counts) if raw_token in token_counts else model_token)
    return pred

def pred_mfr_confidence(row, threshold, counts_by_lang=validation_counts_by_lang):
    token_counts = counts_by_lang.get(row['lang'], {})
    pred = []
    for raw_token, model_token in zip(row['raw'], row['pred']):
        if mfr_token_confidence(raw_token, token_counts) >= threshold:
            pred.append(mfr_token_prediction(raw_token, token_counts))
        else:
            pred.append(model_token)
    return pred

def pred_lang_route(row, mfr_langs, counts_by_lang=validation_counts_by_lang):
    if row['lang'] in mfr_langs:
        return pred_mfr_known(row, counts_by_lang)
    return list(row['pred'])

def prefer_raw_for_case_only_changes(row):
    pred = []
    for raw_token, model_token in zip(row['raw'], row['pred']):
        if raw_token.lower() == model_token.lower() and raw_token != model_token:
            pred.append(raw_token)
        else:
            pred.append(model_token)
    return pred


## 12. Sweep Fast Offline Experiments


In [ ]:
experiments = []

experiments.append(('model', validation_model_records))
experiments.append(('raw', clone_with_pred(validation_model_records, pred_raw)))
experiments.append(('mfr', clone_with_pred(validation_model_records, pred_mfr)))
experiments.append(('mfr-known', clone_with_pred(validation_model_records, pred_mfr_known)))
experiments.append(('postprocess-case-only-to-raw', clone_with_pred(validation_model_records, prefer_raw_for_case_only_changes)))

for threshold in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00]:
    name = f'mfr-confidence-{threshold:.2f}'
    records = clone_with_pred(validation_model_records, lambda row, t=threshold: pred_mfr_confidence(row, t))
    experiments.append((name, records))

lang_lists = {
    'none': [],
    'original-id-iden-sr': ['id', 'iden', 'sr'],
    'de-only': ['de'],
    'id-iden-sr-de': ['id', 'iden', 'sr', 'de'],
    'id-iden-sr-da': ['id', 'iden', 'sr', 'da'],
}
for label, langs in lang_lists.items():
    mfr_langs = set(langs)
    records = clone_with_pred(validation_model_records, lambda row, s=mfr_langs: pred_lang_route(row, s))
    experiments.append((f'lang-route-{label}', records))

ranking = sorted([score_row(name, records) for name, records in experiments], key=lambda row: row['err'], reverse=True)
print(f"{'experiment':35s} {'LAI':>8s} {'Acc':>8s} {'ERR':>8s}")
print('-' * 64)
for row in ranking:
    print(f"{row['name']:35s} {row['lai']*100:8.2f} {row['accuracy']*100:8.2f} {row['err']*100:8.2f}")

BEST_EXPERIMENT_NAME = ranking[0]['name']
BEST_VALIDATION_RECORDS = dict(experiments)[BEST_EXPERIMENT_NAME]
BEST_VALIDATION_PATH = save_records(BEST_VALIDATION_RECORDS, OUTPUT_ROOT / f'best_validation_{BEST_EXPERIMENT_NAME}.json')
(OUTPUT_ROOT / 'offline_validation_ranking.json').write_text(json.dumps(ranking, indent=2), encoding='utf-8')

print('\nBEST_EXPERIMENT_NAME:', BEST_EXPERIMENT_NAME)
print('BEST_VALIDATION_PATH:', BEST_VALIDATION_PATH)


## 13. Per-Language Diagnostics


In [ ]:
from collections import defaultdict

def score_by_language(records):
    groups = defaultdict(list)
    for row in records:
        groups[row['lang']].append(row)
    rows = []
    for lang, lang_records in groups.items():
        lai, acc, err = score_records(lang_records, info=False)
        rows.append({'lang': lang, 'sentences': len(lang_records), 'lai': lai, 'accuracy': acc, 'err': err})
    return sorted(rows, key=lambda row: row['err'], reverse=True)

for experiment_name in ['model', 'mfr', 'mfr-known', BEST_EXPERIMENT_NAME]:
    records = dict(experiments)[experiment_name]
    print('\n', experiment_name)
    print(f"{'lang':8s} {'sent':>6s} {'LAI':>8s} {'Acc':>8s} {'ERR':>8s}")
    for row in score_by_language(records):
        print(f"{row['lang']:8s} {row['sentences']:6d} {row['lai']*100:8.2f} {row['accuracy']*100:8.2f} {row['err']*100:8.2f}")


## 14. Optional Manual Strategy Selection


In [ ]:
# Examples: 'model', 'mfr-confidence-0.90', 'lang-route-original-id-iden-sr'
SELECTED_EXPERIMENT_NAME = BEST_EXPERIMENT_NAME
print('Selected:', SELECTED_EXPERIMENT_NAME)


## 15. Create Or Reuse Test Model Predictions


In [ ]:
BASELINE_TEST_MODEL_PREDICTIONS = (
    BASELINE_RUN_OUTPUT_ROOT / 'test_predictions' / f'{GEMMA4_MODEL_KEY}_model_{RUN_ID}' / 'predictions.json'
)
TEST_MODEL_DIR = TEST_OUTPUT_ROOT / 'test_model_once'
TEST_MODEL_PREDICTIONS = (
    BASELINE_TEST_MODEL_PREDICTIONS
    if BASELINE_TEST_MODEL_PREDICTIONS.exists()
    else TEST_MODEL_DIR / 'predictions.json'
)

if not TEST_MODEL_PREDICTIONS.exists():
    run_cmd([
        sys.executable,
        'predict_lora.py',
        '--adapter-dir', FINAL_ADAPTER_DIR,
        '--model-id', MODEL_ID,
        '--dataset-id', dataset_arg(),
        '--split', 'test',
        '--output-dir', TEST_MODEL_DIR,
        '--hf-token', HF_TOKEN,
        '--quantization', '4bit',
        '--prediction-strategy', 'model',
        '--fallback', 'mfr',
        '--max-new-tokens', '96',
        '--batch-size', '0',
        '--progress-steps', '256',
        '--preview-examples', '5',
    ])
else:
    print('Using cached test model predictions:', TEST_MODEL_PREDICTIONS)

test_model_records = load_records(TEST_MODEL_PREDICTIONS)


## 16. Apply Selected Strategy And Build Basic ZIP


In [ ]:
test_counts_by_lang = make_mfr_counts(data, 'test')

def apply_named_strategy(records, name, counts_by_lang):
    if name == 'model':
        return [dict(row) for row in records]
    if name == 'raw':
        return clone_with_pred(records, pred_raw)
    if name == 'mfr':
        return clone_with_pred(records, lambda row: pred_mfr(row, counts_by_lang))
    if name == 'mfr-known':
        return clone_with_pred(records, lambda row: pred_mfr_known(row, counts_by_lang))
    if name == 'postprocess-case-only-to-raw':
        return clone_with_pred(records, prefer_raw_for_case_only_changes)
    if name.startswith('mfr-confidence-'):
        threshold = float(name.removeprefix('mfr-confidence-'))
        return clone_with_pred(records, lambda row: pred_mfr_confidence(row, threshold, counts_by_lang))
    if name.startswith('lang-route-'):
        label = name.removeprefix('lang-route-')
        if label not in lang_lists:
            raise ValueError(f'Unknown language route label: {label}')
        mfr_langs = set(lang_lists[label])
        return clone_with_pred(records, lambda row: pred_lang_route(row, mfr_langs, counts_by_lang))
    raise ValueError(f'Unknown strategy: {name}')

final_records = apply_named_strategy(test_model_records, SELECTED_EXPERIMENT_NAME, test_counts_by_lang)
for idx, row in enumerate(final_records):
    if set(row) != {'raw', 'norm', 'lang', 'pred'}:
        raise ValueError(f'Bad keys at row {idx}: {row.keys()}')
    if len(row['raw']) != len(row['norm']) or len(row['raw']) != len(row['pred']):
        raise ValueError(f'Length mismatch at row {idx}')

final_predictions_path = save_records(final_records, OUTPUT_ROOT / f'test_predictions_{SELECTED_EXPERIMENT_NAME}.json')
final_zip = make_zip(final_predictions_path, OUTPUT_ROOT / f'codabench_adapter_reuse_{SELECTED_EXPERIMENT_NAME}_{EXPERIMENT_ID}.zip')

print('Final predictions:', final_predictions_path)
print('Upload this ZIP:', final_zip)
print('Records:', len(final_records))
print('First record:', final_records[0])


## 17. Advanced Typed Router And ZIP


In [ ]:
# ADVANCED_TYPED_ROUTER_CELL_V1
from collections import Counter, defaultdict
from pathlib import Path
import itertools
import json

ADVANCED_OUTPUT_ROOT = OUTPUT_ROOT / 'advanced_max_score'
ADVANCED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

required_names = [
    'validation_model_records',
    'data',
    'make_mfr_counts',
    'save_records',
    'make_zip',
    'OUTPUT_ROOT',
    'EXPERIMENT_ID',
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f'Run the earlier notebook sections first. Missing: {missing}')

if 'test_model_records' not in globals():
    if 'TEST_MODEL_PREDICTIONS' in globals() and Path(TEST_MODEL_PREDICTIONS).exists():
        test_model_records = load_records(TEST_MODEL_PREDICTIONS)
    else:
        raise RuntimeError('Run Section 10 first so test_model_records is available.')

validation_counts_by_lang = make_mfr_counts(data, 'validation')  # train only
test_counts_by_lang = make_mfr_counts(data, 'test')              # train + validation


def _top_mfr(raw_token, token_counts):
    if raw_token not in token_counts:
        return raw_token, 0.0, 'unseen'
    replacements = token_counts[raw_token]
    total = sum(replacements.values())
    if total <= 0:
        return raw_token, 0.0, 'unseen'
    top = max(replacements, key=replacements.get)
    conf = replacements[top] / total
    if top == raw_token:
        kind = 'identity'
    elif top == '':
        kind = 'deletion'
    elif ' ' in top:
        kind = 'multiword'
    else:
        kind = 'replacement'
    return top, conf, kind


def _flatten_validation(records, counts_by_lang):
    tokens = []
    for sent_idx, row in enumerate(records):
        token_counts = counts_by_lang.get(row['lang'], {})
        for tok_idx, (raw, gold, model) in enumerate(zip(row['raw'], row['norm'], row['pred'])):
            mfr_top, mfr_conf, mfr_kind = _top_mfr(raw, token_counts)
            tokens.append({
                'sent_idx': sent_idx,
                'tok_idx': tok_idx,
                'lang': row['lang'],
                'raw': raw,
                'gold': gold,
                'model': model,
                'mfr': mfr_top,
                'conf': mfr_conf,
                'kind': mfr_kind,
            })
    return tokens


def _metrics_from_correct(tokens, correct):
    total = len(tokens)
    changed = sum(tok['raw'] != tok['gold'] for tok in tokens)
    lai = (total - changed) / total
    accuracy = correct / total
    err = (accuracy - lai) / (1 - lai) if changed else 0.0
    return {'lai': lai, 'accuracy': accuracy, 'err': err, 'correct': correct, 'total': total}


def _config_name(cfg):
    return (
        f"typed_i{cfg['identity']:.2f}_r{cfg['replacement']:.2f}_"
        f"d{cfg['deletion']:.2f}_m{cfg['multiword']:.2f}"
    )


def _config_pred(tok, cfg):
    kind = tok['kind']
    if kind != 'unseen' and tok['conf'] >= cfg.get(kind, 1.01):
        return tok['mfr']
    return tok['model']


def _score_config(tokens, cfg):
    correct = sum(_config_pred(tok, cfg) == tok['gold'] for tok in tokens)
    return _metrics_from_correct(tokens, correct)


def _score_lang_configs(tokens, lang_cfg, default_cfg, token_policy=None):
    correct = 0
    token_policy = token_policy or {}
    for tok in tokens:
        cfg = lang_cfg.get(tok['lang'], default_cfg)
        action = token_policy.get((tok['lang'], tok['raw']))
        pred = _action_pred(tok, action, cfg) if action else _config_pred(tok, cfg)
        correct += pred == tok['gold']
    return _metrics_from_correct(tokens, correct)


def _action_pred(tok, action, cfg):
    if action == 'model':
        return tok['model']
    if action == 'raw':
        return tok['raw']
    if action == 'mfr':
        return tok['mfr']
    return _config_pred(tok, cfg)


validation_tokens = _flatten_validation(validation_model_records, validation_counts_by_lang)
tokens_by_lang = defaultdict(list)
for tok in validation_tokens:
    tokens_by_lang[tok['lang']].append(tok)

# 1. Search typed MFR thresholds. 1.01 means "disable this MFR type".
grid = {
    'identity': [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00, 1.01],
    'replacement': [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00, 1.01],
    'deletion': [0.50, 0.70, 0.90, 1.00, 1.01],
    'multiword': [0.50, 0.70, 0.90, 1.00, 1.01],
}
configs = []
for values in itertools.product(*(grid[key] for key in ['identity', 'replacement', 'deletion', 'multiword'])):
    cfg = dict(zip(['identity', 'replacement', 'deletion', 'multiword'], values))
    cfg['name'] = _config_name(cfg)
    configs.append(cfg)

global_ranking = []
for cfg in configs:
    row = _score_config(validation_tokens, cfg)
    row['name'] = cfg['name']
    row['config'] = cfg
    global_ranking.append(row)
global_ranking.sort(key=lambda row: (row['err'], row['accuracy']), reverse=True)
global_best_cfg = global_ranking[0]['config']

print('Top global typed-threshold configs')
print(f"{'rank':>4s} {'ERR':>8s} {'Acc':>8s}  config")
for rank, row in enumerate(global_ranking[:12], start=1):
    print(f"{rank:4d} {row['err']*100:8.2f} {row['accuracy']*100:8.2f}  {row['name']}")

# 2. Choose the best typed config separately for each validation language.
per_lang_cfg = {}
print('\nBest typed config per language')
print(f"{'lang':8s} {'ERR':>8s} {'Acc':>8s}  config")
for lang, lang_tokens in sorted(tokens_by_lang.items()):
    best = None
    for cfg in configs:
        row = _score_config(lang_tokens, cfg)
        if best is None or (row['err'], row['accuracy']) > (best['err'], best['accuracy']):
            best = {**row, 'config': cfg}
    per_lang_cfg[lang] = best['config']
    print(f"{lang:8s} {best['err']*100:8.2f} {best['accuracy']*100:8.2f}  {best['config']['name']}")

per_lang_score = _score_lang_configs(validation_tokens, per_lang_cfg, global_best_cfg)
print(
    '\nPer-language typed router validation: '
    f"LAI={per_lang_score['lai']*100:.2f} "
    f"Acc={per_lang_score['accuracy']*100:.2f} "
    f"ERR={per_lang_score['err']*100:.2f}"
)

# 3. Learn exact raw-token actions where validation repeatedly shows that model/raw/MFR
# beats the per-language typed default. This is powerful but more overfit.
USE_TOKEN_ACTION_POLICY = True
MIN_POLICY_SUPPORT = 2
MIN_POLICY_MARGIN = 1
MIN_POLICY_PRECISION = 0.67

policy_stats = defaultdict(lambda: {
    'n': 0,
    'default_correct': 0,
    'correct': Counter(),
})
for tok in validation_tokens:
    cfg = per_lang_cfg.get(tok['lang'], global_best_cfg)
    key = (tok['lang'], tok['raw'])
    stat = policy_stats[key]
    stat['n'] += 1
    default_pred = _config_pred(tok, cfg)
    stat['default_correct'] += default_pred == tok['gold']
    for action in ['model', 'raw', 'mfr']:
        stat['correct'][action] += _action_pred(tok, action, cfg) == tok['gold']

token_policy = {}
for key, stat in policy_stats.items():
    n = stat['n']
    if n < MIN_POLICY_SUPPORT:
        continue
    action, best_correct = stat['correct'].most_common(1)[0]
    margin = best_correct - stat['default_correct']
    precision = best_correct / n
    if margin >= MIN_POLICY_MARGIN and precision >= MIN_POLICY_PRECISION:
        token_policy[key] = action

policy_for_score = token_policy if USE_TOKEN_ACTION_POLICY else {}
policy_score = _score_lang_configs(validation_tokens, per_lang_cfg, global_best_cfg, policy_for_score)
print(
    '\nPer-language typed router + token action policy validation '
    '(optimistic): '
    f"LAI={policy_score['lai']*100:.2f} "
    f"Acc={policy_score['accuracy']*100:.2f} "
    f"ERR={policy_score['err']*100:.2f}"
)
print('Token action policy entries:', len(token_policy))
print('Token action policy action counts:', dict(Counter(token_policy.values())))

# 4. Literal validation memory for final test only.
# This uses public validation labels as a conservative dictionary when a raw token
# has a consistent validation normalization. It is not included in the validation
# score above because scoring it on the same validation tokens would be leakage.
USE_VALIDATION_LITERAL_MEMORY = True
LITERAL_MEMORY_MIN_SUPPORT = 2
LITERAL_MEMORY_MIN_CONFIDENCE = 1.0

literal_counts = defaultdict(Counter)
for row in validation_model_records:
    for raw, gold in zip(row['raw'], row['norm']):
        literal_counts[(row['lang'], raw)][gold] += 1

literal_memory = {}
for key, counts in literal_counts.items():
    total = sum(counts.values())
    gold, count = counts.most_common(1)[0]
    if total >= LITERAL_MEMORY_MIN_SUPPORT and count / total >= LITERAL_MEMORY_MIN_CONFIDENCE:
        literal_memory[key] = gold

print('Validation literal-memory entries for final test:', len(literal_memory))


def _apply_advanced_router(records, counts_by_lang, lang_cfg, default_cfg, token_policy, literal_memory):
    out = []
    for row in records:
        cfg = lang_cfg.get(row['lang'], default_cfg)
        token_counts = counts_by_lang.get(row['lang'], {})
        pred = []
        for raw, model in zip(row['raw'], row['pred']):
            literal_key = (row['lang'], raw)
            if USE_VALIDATION_LITERAL_MEMORY and literal_key in literal_memory:
                pred.append(literal_memory[literal_key])
                continue

            mfr_top, mfr_conf, mfr_kind = _top_mfr(raw, token_counts)
            tok = {
                'lang': row['lang'],
                'raw': raw,
                'model': model,
                'mfr': mfr_top,
                'conf': mfr_conf,
                'kind': mfr_kind,
            }
            action = token_policy.get((row['lang'], raw))
            pred.append(_action_pred(tok, action, cfg) if action else _config_pred(tok, cfg))

        new_row = dict(row)
        new_row['pred'] = pred
        out.append(new_row)
    return out


selected_policy = token_policy if USE_TOKEN_ACTION_POLICY else {}
advanced_records = _apply_advanced_router(
    test_model_records,
    test_counts_by_lang,
    per_lang_cfg,
    global_best_cfg,
    selected_policy,
    literal_memory,
)

for idx, row in enumerate(advanced_records):
    if set(row) != {'raw', 'norm', 'lang', 'pred'}:
        raise ValueError(f'Bad keys at row {idx}: {row.keys()}')
    if len(row['raw']) != len(row['norm']) or len(row['raw']) != len(row['pred']):
        raise ValueError(f'Length mismatch at row {idx}')

advanced_config = {
    'global_best_config': global_best_cfg,
    'per_lang_config': per_lang_cfg,
    'use_token_action_policy': USE_TOKEN_ACTION_POLICY,
    'token_action_policy_size': len(token_policy),
    'use_validation_literal_memory': USE_VALIDATION_LITERAL_MEMORY,
    'literal_memory_size': len(literal_memory),
    'per_lang_validation_score': per_lang_score,
    'policy_validation_score_optimistic': policy_score,
}
(ADVANCED_OUTPUT_ROOT / 'advanced_router_config.json').write_text(
    json.dumps(advanced_config, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

advanced_predictions_path = save_records(
    advanced_records,
    ADVANCED_OUTPUT_ROOT / f'test_predictions_advanced_typed_router_{EXPERIMENT_ID}.json',
)
advanced_zip = make_zip(
    advanced_predictions_path,
    ADVANCED_OUTPUT_ROOT / f'codabench_advanced_typed_router_{EXPERIMENT_ID}.zip',
)

print('\nAdvanced final predictions:', advanced_predictions_path)
print('Upload this ZIP:', advanced_zip)
print('Records:', len(advanced_records))
print('First record:', advanced_records[0])


## 18. Final ZIP Summary


In [ ]:
created = []
for name in ["advanced_zip", "final_zip"]:
    if name in globals():
        path = Path(globals()[name])
        created.append(path)
        print(f"{name}: {path}")
        if path.exists():
            with zipfile.ZipFile(path) as zf:
                print("  ZIP contents:", zf.namelist())

if not created:
    print("No final ZIP variable was found. Run the test prediction and ZIP cells above.")
else:
    print("Latest upload candidate:", created[0])
